In [14]:
!pip -q install dagshub mlflow

In [15]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="karev23",
    repo_name="ML-final",
    mlflow=True,
)

mlflow.set_experiment("NBEATS_Training")

while mlflow.active_run() is not None:
    mlflow.end_run()

print("Tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "karev23/ML-final"

Repository karev23/ML-final initialized!

🏃 View run NBEATS_Workflow at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/e9aed8225a6f4845b75c82f99a789f93
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7
Tracking URI: https://dagshub.com/karev23/ML-final.mlflow


In [16]:
!pip -q install torch neuralforecast

In [17]:
!pip -q install dagshub mlflow torch neuralforecast

# N-BEATS Training

N-BEATS experiment notebook for the Walmart sales forecasting project using Nixtla `neuralforecast`.

What it does:
- prepares the Walmart panel in Nixtla long format;
- trains a global N-BEATS-style model across all Store-Dept series;
- uses WMAE over the same 39-week expanding validation folds as every other model;
- logs MLflow stages for cleaning, feature selection, cross-validation, HPO, and final output;
- includes notes for comparing N-BEATS against DLinear.

If `NBEATSx` is installed, the notebook uses it with exogenous features. Otherwise it falls back to plain `NBEATS` without exogenous variables.

In [18]:
# Colab/runtime bootstrap. Run once at the start of a fresh runtime.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import glob
    import os
    import subprocess
    import sys
    import zipfile

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "neuralforecast>=1.7,<2",
        "mlflow",
        "dagshub",
        "kaggle",
        "python-dotenv",
        "scikit-learn",
        "pandas",
        "pyarrow",
    ])

    if os.path.exists("dataloader.py") and os.path.exists("features.py"):
        pass
    elif os.path.exists("../dataloader.py") and os.path.exists("../features.py"):
        os.chdir("..")
    elif os.path.isdir("ML-final") and os.path.exists("ML-final/dataloader.py"):
        os.chdir("ML-final")
    else:
        subprocess.check_call(["git", "clone", "https://github.com/ketevan614/ML-final.git"])
        os.chdir("ML-final")

    try:
        from google.colab import userdata
        kaggle_json = userdata.get("KAGGLE_JSON")
        kaggle_username = userdata.get("KAGGLE_USERNAME")
        kaggle_key = userdata.get("KAGGLE_KEY")
        if kaggle_json:
            kaggle_dir = os.path.expanduser("~/.kaggle")
            os.makedirs(kaggle_dir, exist_ok=True)
            kaggle_path = os.path.join(kaggle_dir, "kaggle.json")
            with open(kaggle_path, "w") as f:
                f.write(kaggle_json)
            os.chmod(kaggle_path, 0o600)
        elif kaggle_username and kaggle_key:
            os.environ["KAGGLE_USERNAME"] = kaggle_username
            os.environ["KAGGLE_KEY"] = kaggle_key
    except Exception as exc:
        print(f"[Colab setup warning] Kaggle credentials not configured: {type(exc).__name__}: {exc}")

    if not os.path.exists("data/train.csv"):
        os.makedirs("data", exist_ok=True)
        try:
            subprocess.check_call([
                "kaggle", "competitions", "download",
                "-c", "walmart-recruiting-store-sales-forecasting",
                "-p", "data",
            ])
            for zip_path in glob.glob("data/*.zip"):
                with zipfile.ZipFile(zip_path) as zf:
                    zf.extractall("data")
                os.remove(zip_path)
        except Exception as exc:
            print(f"[Colab setup warning] Kaggle download failed: {type(exc).__name__}: {exc}")

print("IN_COLAB =", IN_COLAB)


[Colab setup warning] Kaggle credentials not configured: TimeoutException: Requesting secret KAGGLE_JSON timed out. Secrets can only be fetched when running from the Colab UI.
IN_COLAB = True


In [19]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "dataloader.py").exists() and (candidate / "features.py").exists():
            return candidate
    content_dir = Path("/content")
    if content_dir.exists():
        for candidate in content_dir.glob("**/dataloader.py"):
            root = candidate.parent
            if (root / "features.py").exists():
                return root
    raise FileNotFoundError(
        "Could not find the project root. In Colab, upload/clone the full repo and cd into it before running."
    )

ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT))

from dataloader import load_raw, merge_all, make_submission
from features import build_features
from metrics import wmae
from validation import expanding_window_folds

warnings.filterwarnings("ignore")
DATA_DIR = ROOT / "data"
(ROOT / "models").mkdir(exist_ok=True)
required_data_files = ["train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"]
missing_data_files = [name for name in required_data_files if not (DATA_DIR / name).exists()]
if missing_data_files:
    raise FileNotFoundError(f"Missing data files in {DATA_DIR}: {missing_data_files}")
EXPERIMENT_NAME = "NBEATS_Training"
MODEL_ALIAS = "NBEATS"
HORIZON = 39
RANDOM_STATE = 42


In [20]:
import os
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=True)
except ImportError:
    pass

from neuralforecast import NeuralForecast
from neuralforecast_pyfunc import NeuralForecastRawPyFunc
from neuralforecast.models import NBEATS as NBEATSModel

# Use plain NBEATS as a target-history model. NBEATSx requires a complete
# future exogenous dataframe for every Store-Dept/date pair, which the Walmart
# panel does not always provide cleanly in validation folds.
SUPPORTS_EXOG = False

# Some local environments set a dead proxy (for example 127.0.0.1:9),
# which prevents MLflow from reaching DagsHub. Clear it before tracking calls.
for key in [
    "HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy",
    "ALL_PROXY", "all_proxy",
]:
    os.environ.pop(key, None)

mlflow.set_experiment(EXPERIMENT_NAME)

# Colab notebooks are often rerun from the middle. Close stale runs so reruns
# do not fail with "run is already active".
while mlflow.active_run() is not None:
    mlflow.end_run()


# Do not let temporary DagsHub/DNS failures crash long training cells.
# Failed log calls are printed; rerun the cell later on a stable connection if a required metric is missing remotely.
def _safe_mlflow_call(fn, label):
    def wrapped(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            print(f"[MLflow warning] {label} failed: {type(exc).__name__}: {str(exc)[:300]}")
            return None
    return wrapped

mlflow.log_param = _safe_mlflow_call(mlflow.log_param, "log_param")
mlflow.log_params = _safe_mlflow_call(mlflow.log_params, "log_params")
mlflow.log_metric = _safe_mlflow_call(mlflow.log_metric, "log_metric")
mlflow.log_metrics = _safe_mlflow_call(mlflow.log_metrics, "log_metrics")
mlflow.log_artifact = _safe_mlflow_call(mlflow.log_artifact, "log_artifact")
mlflow.log_artifacts = _safe_mlflow_call(mlflow.log_artifacts, "log_artifacts")

# Run this cell once before the stage cells when you want MLflow to group
# the required Cleaning -> Feature Selection -> CV -> HPO -> Final runs.
parent_run = mlflow.start_run(run_name="NBEATS_Workflow")

In [21]:
train_raw, test_raw, features_df, stores_df, sample_submission = load_raw(DATA_DIR)
train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

def to_nixtla(raw_frame: pd.DataFrame, history_frame: pd.DataFrame | None = None, include_y: bool = True) -> pd.DataFrame:
    nf = pd.DataFrame({
        "unique_id": raw_frame["Store"].astype(str) + "_" + raw_frame["Dept"].astype(str),
        "ds": pd.to_datetime(raw_frame["Date"]),
    })
    if include_y and "Weekly_Sales" in raw_frame.columns:
        nf["y"] = raw_frame["Weekly_Sales"].astype(float).to_numpy()
    return nf

def holiday_lookup(raw_frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "unique_id": raw_frame["Store"].astype(str) + "_" + raw_frame["Dept"].astype(str),
        "ds": pd.to_datetime(raw_frame["Date"]),
        "IsHoliday_eval": raw_frame["IsHoliday"].astype(bool).to_numpy(),
    })

def normalize_nf_predictions(preds: pd.DataFrame) -> pd.DataFrame:
    preds = preds.reset_index()
    unnamed = [c for c in preds.columns if str(c).startswith("level_") or str(c) == "index"]
    if unnamed:
        preds = preds.drop(columns=unnamed)
    return preds


## MLflow Stage: Cleaning

N-BEATS is more expensive than DLinear. Keep all series first, but log series lengths so the README can explain if a later run filters very sparse series.

In [22]:
with mlflow.start_run(run_name="NBEATS_Cleaning", nested=True):
    series_lengths = train_raw.groupby(["Store", "Dept"]).size()
    mlflow.log_param("negative_sales_training_policy", "keep")
    mlflow.log_param("negative_prediction_policy", "clip_to_zero_for_submission")
    mlflow.log_param("short_series_policy", "keep_initially")
    mlflow.log_metric("series_count", int(series_lengths.shape[0]))
    mlflow.log_metric("min_series_length", int(series_lengths.min()))
    mlflow.log_metric("median_series_length", float(series_lengths.median()))

🏃 View run NBEATS_Cleaning at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/2476c4c5764141cebc7931e15721728b
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7


## MLflow Stage: Feature Selection

Plain N-BEATS does not use future exogenous features. NBEATSx does, so the notebook switches behavior based on the installed class.

In [23]:
ALL_EXOG = []
FUTR_EXOG_LIST = []

with mlflow.start_run(run_name="NBEATS_Feature_Selection", nested=True):
    mlflow.log_param("model_class", NBEATSModel.__name__)
    mlflow.log_param("supports_exogenous", SUPPORTS_EXOG)
    mlflow.log_metric("future_exog_count", len(FUTR_EXOG_LIST))
    pd.Series(FUTR_EXOG_LIST, name="feature").to_csv(ROOT / "models" / "nbeats_exog_features.csv", index=False)
    mlflow.log_artifact(str(ROOT / "models" / "nbeats_exog_features.csv"))


🏃 View run NBEATS_Feature_Selection at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/1dc11b32bfa248fab03a711b7836f383
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7


In [24]:
def build_nbeats(input_size: int = 52, max_steps: int = 500, stack_types=None):
    input_size = int(input_size)
    max_steps = int(max_steps)
    stack_types = stack_types or ["trend", "seasonality"]
    return NBEATSModel(
        h=HORIZON,
        input_size=input_size,
        max_steps=max_steps,
        learning_rate=1e-3,
        alias=MODEL_ALIAS,
        random_seed=RANDOM_STATE,
        start_padding_enabled=True,
        stack_types=stack_types,
    )

def cv_nbeats(input_size: int = 52, max_steps: int = 500) -> list[float]:
    scores = []
    for fold, (tr_mask, val_mask) in enumerate(expanding_window_folds(train_raw["Date"], n_splits=3, horizon=HORIZON), start=1):
        fold_train_raw = train_raw.loc[tr_mask].copy()
        fold_val_raw = train_raw.loc[val_mask].copy()
        nf_train = to_nixtla(fold_train_raw)
        nf_val = to_nixtla(fold_val_raw, history_frame=fold_train_raw)
        nf = NeuralForecast(models=[build_nbeats(input_size=input_size, max_steps=max_steps)], freq="W-FRI")
        nf.fit(df=nf_train)
        preds = normalize_nf_predictions(nf.predict())
        scored = nf_val[["unique_id", "ds", "y"]].merge(preds[["unique_id", "ds", MODEL_ALIAS]], on=["unique_id", "ds"], how="left")
        scored = scored.merge(holiday_lookup(fold_val_raw), on=["unique_id", "ds"], how="left")
        scored = scored.dropna(subset=[MODEL_ALIAS])
        pred = np.clip(scored[MODEL_ALIAS].to_numpy(), 0, None)
        scores.append(wmae(scored["y"], pred, scored["IsHoliday_eval"]))
    return scores


## MLflow Stage: Cross-Validation

In [25]:
with mlflow.start_run(run_name="NBEATS_CrossValidation", nested=True):
    cv_scores = cv_nbeats(input_size=52, max_steps=500)
    for i, score in enumerate(cv_scores, start=1):
        mlflow.log_metric(f"fold_{i}_wmae", score)
    mlflow.log_metric("mean_cv_wmae", float(np.mean(cv_scores)))


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 1.7 M  | train
-------------------------------------------------------
1.7 M     Trainable params
7.2 K     Non-trainable params
1.7 M     Total params
6.874     Total estimated model params size (MB)
22        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

🏃 View run NBEATS_CrossValidation at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/7c5495e251aa494c9f79dcb7c32ad4c4
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7


## MLflow Stage: HPO

Tune only a small grid at first. N-BEATS is slower than DLinear, so the report should emphasize whether the extra cost improved WMAE.

In [26]:
  SEARCH_SPACE = [
    {"input_size": 26, "max_steps": 500},
    {"input_size": 52, "max_steps": 500},
    {"input_size": 78, "max_steps": 800},
]

with mlflow.start_run(run_name="NBEATS_HPO", nested=True):
    results = []
    for params in SEARCH_SPACE:
        scores = cv_nbeats(**params)
        results.append({**params, "mean_wmae": float(np.mean(scores))})
    hpo_results = pd.DataFrame(results).sort_values("mean_wmae")
    best_params = hpo_results.iloc[0].drop("mean_wmae").to_dict()
    hpo_results.to_csv(ROOT / "models" / "nbeats_hpo_results.csv", index=False)
    mlflow.log_params(best_params)
    mlflow.log_metric("best_mean_cv_wmae", float(hpo_results.iloc[0]["mean_wmae"]))
    mlflow.log_artifact(str(ROOT / "models" / "nbeats_hpo_results.csv"))


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 1.7 M  | train
-------------------------------------------------------
1.7 M     Trainable params
5.1 K     Non-trainable params
1.7 M     Total params
6.759     Total estimated model params size (MB)
22        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=800` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

🏃 View run NBEATS_HPO at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/1a9c2b9572b8447ba96ca9d925936dab
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7


## MLflow Stage: Final Model

In [27]:
with mlflow.start_run(run_name="NBEATS_Final", nested=True):
    final_params = globals().get("best_params", {"input_size": 52, "max_steps": 500})
    nf_train = to_nixtla(train_raw)
    nf_test = to_nixtla(test_raw, history_frame=train_raw, include_y=False)
    final_nf = NeuralForecast(models=[build_nbeats(**final_params)], freq="W-FRI")
    final_nf.fit(df=nf_train)

    test_preds = normalize_nf_predictions(final_nf.predict())
    scored_test = nf_test.merge(test_preds[["unique_id", "ds", MODEL_ALIAS]], on=["unique_id", "ds"], how="left")
    fallback = float(train_raw["Weekly_Sales"].mean())
    test_pred_values = np.clip(scored_test[MODEL_ALIAS].fillna(fallback).to_numpy(), 0, None)
    make_submission(test_raw, test_pred_values, ROOT / "models" / "nbeats_submission.csv")

    mlflow.log_params(final_params)
    mlflow.log_artifact(str(ROOT / "models" / "nbeats_submission.csv"))
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=NeuralForecastRawPyFunc(
            neuralforecast_model=final_nf,
            features_df=features_df,
            stores_df=stores_df,
            train_history_raw=train_raw,
            model_alias=MODEL_ALIAS,
            uses_exog=False,
        ),
        code_paths=[
            str(ROOT / "neuralforecast_pyfunc.py"),
            str(ROOT / "dataloader.py"),
            str(ROOT / "features.py"),
        ],
        pip_requirements=[
            "mlflow",
            "numpy",
            "pandas",
            "scikit-learn",
            "torch",
            "neuralforecast>=1.7,<2",
        ],
    )

mlflow.end_run()


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 1.7 M  | train
-------------------------------------------------------
1.7 M     Trainable params
7.2 K     Non-trainable params
1.7 M     Total params
6.874     Total estimated model params size (MB)
22        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

2026/07/12 18:34:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 18:34:31 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/12 18:34:31 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


🏃 View run NBEATS_Final at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/06ab152fbdae433eb57f1ee47b554203
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7
🏃 View run NBEATS_Workflow at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7/runs/8f8af03e6d3f47b5b83953e4ff8cd3d1
🧪 View experiment at: https://dagshub.com/karev23/ML-final.mlflow/#/experiments/7


## README Takeaways

- N-BEATS learns global nonlinear trend and seasonality patterns across Store-Dept series.
- It should be compared directly with DLinear: if WMAE is not meaningfully lower, the added training cost is hard to justify.
- If plain N-BEATS underperforms trees, a likely reason is that sharp holiday and markdown effects are easier for boosted trees to capture from tabular features.